# Práctica 1 – Notebook 2: Predicciones con el Modelo Final

**Asignatura:** Aprendizaje Automático 2025-26  
**Grupo:**
- Pablo García Aparicio
- Miguel Merino Sánchez

**NIA:** 100522190  

Este notebook carga el modelo final entrenado (`modelo_final.joblib`) y lo usa para:
1. Generar predicciones sobre el dataset de competición (`bank_competition.pkl`)
2. Guardar las predicciones en `predicciones.csv`


In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import joblib
from sklearn.preprocessing import LabelEncoder

# Semilla reproducibilidad
SEED = 100522190
np.random.seed(SEED)

print('Librerías cargadas.')

Librerías cargadas.


## 1. Cargar modelo final

In [2]:
modelo = joblib.load('modelo_final.joblib')
print(f'Modelo cargado: {type(modelo.named_steps["clf"]).__name__}')
print(modelo)

Modelo cargado: RandomForestClassifier
Pipeline(steps=[('pre',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'balance', 'day',
                                                   'duration', 'campaign',
                                                   'pdays', 'previous',
                                                   'pdays_contactado']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent

## 2. Cargar datos de competición y aplicar preprocesamiento de pdays

In [3]:
df_comp = pd.read_pickle('bank_competition.pkl')
print(f'Dataset competición: {df_comp.shape[0]} instancias, {df_comp.shape[1]} variables')
print('Columnas:', list(df_comp.columns))
df_comp.head()

Dataset competición: 162 instancias, 16 variables
Columnas: ['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome']


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome
5553,43,management,married,tertiary,no,78,yes,no,cellular,21,nov,36,1,109,1,other
915,34,housemaid,married,secondary,no,0,yes,no,unknown,30,oct,154,1,-1,0,unknown
7652,54,technician,married,secondary,no,3323,yes,yes,cellular,8,apr,59,3,-1,0,unknown
5065,43,blue-collar,single,primary,no,-399,no,yes,cellular,28,jul,662,3,-1,0,unknown
3338,35,blue-collar,married,secondary,no,262,no,no,cellular,15,mar,427,1,181,3,success


In [4]:
# ── Mismo preprocesamiento de pdays que en el notebook 1 ──────────────────
df_comp['pdays_contactado'] = (df_comp['pdays'] != -1).astype(int)
df_comp['pdays'] = df_comp['pdays'].replace(-1, np.nan)

print('Preprocesamiento pdays aplicado.')
print(f"pdays_contactado (0=no contactado, 1=contactado): {df_comp['pdays_contactado'].value_counts().to_dict()}")

Preprocesamiento pdays aplicado.
pdays_contactado (0=no contactado, 1=contactado): {0: 121, 1: 41}


## 3. Generar predicciones

In [5]:
# Predicciones de clase y probabilidades
y_pred = modelo.predict(df_comp)
y_proba = modelo.predict_proba(df_comp)[:, 1]

# Convertir a etiquetas legibles (yes/no)
# Reconstruir el LabelEncoder con el mismo mapping
le = LabelEncoder()
le.classes_ = np.array(['no', 'yes'])  # orden estándar
y_pred_labels = le.inverse_transform(y_pred)

print(f'Predicciones generadas: {len(y_pred_labels)} instancias')
print(f'Distribución: {pd.Series(y_pred_labels).value_counts().to_dict()}')

Predicciones generadas: 162 instancias
Distribución: {'no': 90, 'yes': 72}


## 4. Guardar predicciones en CSV

In [6]:
pred_df = pd.DataFrame({
    'deposit': y_pred_labels,
    'probabilidad_yes': y_proba.round(4)
})

# Guardar solo la columna deposit (sin probabilidades) para la competición
pred_df[['deposit']].to_csv('predicciones.csv', index=False)
print('Predicciones guardadas en: predicciones.csv')

# Preview
print('\nPrimeras 10 predicciones:')
display(pred_df.head(10))

Predicciones guardadas en: predicciones.csv

Primeras 10 predicciones:


,deposit,probabilidad_yes
0,no,0.1025
1,yes,0.7374
2,no,0.1474
3,yes,0.7376
4,yes,0.9698
5,no,0.0731
6,yes,0.8293
7,no,0.0870
8,yes,0.8721
9,no,0.1430


## 5. Verificación: Predicciones del modelo en 2 instancias específicas

*(Estas mismas instancias se comparan después con la app Streamlit para verificar que las predicciones coinciden)*

In [7]:
# Instancia 1: Primer cliente del dataset de competición
instancia_1 = df_comp.iloc[[0]]
pred_1 = modelo.predict(instancia_1)[0]
proba_1 = modelo.predict_proba(instancia_1)[0, 1]

# Instancia 2: Segundo cliente del dataset de competición
instancia_2 = df_comp.iloc[[1]]
pred_2 = modelo.predict(instancia_2)[0]
proba_2 = modelo.predict_proba(instancia_2)[0, 1]

print('=== Verificación de predicciones ===')
print(f'Instancia 1:')
display(instancia_1)
pred_label_1 = le.inverse_transform([pred_1])[0]
print(f'  → Predicción: {pred_label_1} (Prob. YES={proba_1:.4f})')

print(f'\nInstancia 2:')
display(instancia_2)
pred_label_2 = le.inverse_transform([pred_2])[0]
print(f'  → Predicción: {pred_label_2} (Prob. YES={proba_2:.4f})')

print('\n Estas predicciones deben coincidir exactamente con las de la app Streamlit.')

=== Verificación de predicciones ===
Instancia 1:


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,pdays_contactado
5553,43,management,married,tertiary,no,78,yes,no,cellular,21,nov,36,1,109.0,1,other,1


  → Predicción: no (Prob. YES=0.1025)

Instancia 2:


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,pdays_contactado
915,34,housemaid,married,secondary,no,0,yes,no,unknown,30,oct,154,1,NaN,0,unknown,0


  → Predicción: yes (Prob. YES=0.7374)

 Estas predicciones deben coincidir exactamente con las de la app Streamlit.
